# Phase 3 : Benchmark machine learning contre le tarif GLM

**Objectif.** Mesurer ce qu'un modèle de gradient boosting (LightGBM) apporte au tarif GLM v0.2.0 : sur les mêmes polices de test, avec les mêmes métriques, à niveau de prime égal. La question n'est pas "le GBM fait-il mieux ?" mais "où, de combien, et à quel prix en lisibilité ?".

**Le tarif à battre.** Pour une police $i$, on note :
- $e_i$ son exposition en années, plafonnée à 1 ;
- $\hat\lambda_i$ la fréquence annuelle de sinistres prédite par le GLM Poisson, c'est-à-dire le nombre attendu de sinistres pour une exposition d'un an ;
- $\hat\mu_i^{ecr}$ le coût moyen d'un sinistre indemnisé, écrêté au 99e percentile (16 282 EUR), prédit par le GLM Gamma ;
- $\tau_{obs} = 0.7317$ la proportion de sinistres déclarés qui donnent lieu à un montant ;
- $\tau_{graves} = 0.4529$ le taux de rechargement des sinistres graves, tel que $1 + \tau_{graves}$ ramène la charge écrêtée à la charge totale observée sur le train ;
- $k = 0.994665$ la constante de recalage du niveau global de prime sur le train.

La prime pure annuelle prédite vaut

$$\widehat{PP}_i = k \times \hat\lambda_i \times \tau_{obs} \times \hat\mu_i^{ecr} \times (1 + \tau_{graves})$$

et la charge attendue de la police $i$ vaut $e_i \times \widehat{PP}_i$.

| Indicateur (test) | Valeur GLM |
|---|---|
| Indice de Gini (prime pure) | 0.3592 |
| Lift G10/G1 fréquence | 4.52 |
| D2 fréquence (déviance Poisson) | 0.0357 |
| D2 sévérité (déviance Gamma) | 0.0034 |
| D2 prime pure (déviance Tweedie, p = 1.667) | 0.0519 |
| Calibration A/E train / test | 1.0000 / 0.9531 |
| Amplitude des primes (99e / 1er pct) | 11.9 |

**Règles du benchmark.**
1. *Même découpage, test sanctuarisé.* Aucun choix (hyperparamètres, nombre d'arbres, variables) ne s'appuie sur le test. Tout réglage passe par une validation croisée sur le train.
2. *Mêmes métriques, même code.* Les indicateurs du GLM sont recalculés ici par les fonctions qui évalueront le GBM.
3. *Même niveau de prime.* Le GBM n'a pas la propriété d'équilibre du GLM Poisson à lien canonique. Il est donc recalé sur le train avant toute comparaison.

**Questions à trancher.** L'interaction âge × bonus-malus est-elle un gain réel ou un artefact ? Le résidu non linéaire sur `log_Density` compte-t-il en pratique ? Le plafond de la sévérité vient-il des forfaits ou de la classe de modèles ?

## Journal des décisions (suite de la Phase 2)

| N° | Constat | Décision | Justification | Impact |
|---|---|---|---|---|
| 39 | Le benchmark n'a de sens que si les deux modèles sont jugés sur les mêmes polices | Réutiliser `split_train_test.parquet` tel quel et ne consulter le test qu'à l'évaluation finale | Un réglage guidé par le test (early stopping, hyperparamètres) surestime le GBM, alors que le GLM n'a jamais vu le test | Tout réglage du GBM passe par validation croisée sur le train |
| 40 | LightGBM et XGBoost offrent tous deux les objectifs Poisson, Gamma, Tweedie et un offset | Retenir LightGBM seul | Arbres par histogrammes rapides sur 542 397 lignes, catégorielles natives, empreinte mémoire faible (crash de kernel en Phase 2) | Le récit reste GLM contre GBM, pas un concours de librairies |
| 41 | Le tarif v0.2.0 est figé dans `models/tarif_glm_phase2.json` | Scorer depuis ce fichier plutôt que réajuster les GLM | On compare au tarif tagué, pas à une réestimation, et la même fonction de scoring servira à l'appli en Phase 5 | Fonction de scoring à écrire dans `src/` |
| 42 | `BonusMalusBand` et `VehPowerGrp` sont absentes du parquet et leurs bornes absentes du JSON du tarif | Centraliser toutes les définitions de regroupement dans `src/features.py` | Un tarif doit se calculer à partir des seules variables brutes, et un même code de regroupement évite que notebook et appli appliquent des bornes différentes | Le notebook 03 et l'appli Streamlit importeront la même fonction |
| 43 | `VehGas` est stockée avec ses apostrophes dans les parquets (`'Regular'`) : `load_data.py` ne nettoie que les colonnes de type `object`, alors que pandas 3 lit le texte en type `str`, et la Phase 2 nettoyait en mémoire sans réexporter | Nettoyage idempotent dans `prepare_variables`, et garde-fou au scoring : une modalité absente du tarif lève une erreur | Sans garde-fou, les deux modalités auraient reçu un effet nul, et les polices diesel auraient été tarifées comme la référence essence, sans aucun message | Correction du chargeur reportée à la Phase 6, sans effet en aval |
| 44 | Le regroupement par exposition égale de la Phase 2 excluait la police la plus chère du test, dont la part cumulée valait 1.0000000000001275 | Borner la part cumulée à 1 et trier de façon stable dans `src/evaluation.py` | Le défaut frappe toujours la police la plus chère, là où les modèles diffèrent le plus ; un tri instable rend le Gini dépendant de la machine | Lift de référence du GLM : 4.5248 au lieu de 4.5226 ; tous les lifts de la Phase 3 utilisent la fonction corrigée |
| 45 | Le JSON du tarif ne contient ni la dispersion du GLM sévérité ni l'indice de Tweedie | Recalculer $\hat\phi$ par Pearson à partir des coefficients du JSON (2.0058), en déduire $p = 1.667306$ et le figer pour la phase | La déviance de Tweedie n'est comparable qu'à indice égal | $D^2$ prime pure du GLM reproduit (0.0519) ; même indice pour tous les tarifs |

## 1. Chargement des données et contrôles d'intégrité

Trois vérifications avant toute modélisation :
- **le découpage relu est exactement celui de la Phase 2** (542 397 / 135 616). Si un seul effectif diffère, les chiffres de référence du GLM ne sont plus comparables ;
- **les variables brutes sont présentes** (âge du conducteur, bonus-malus, densité, etc.) à côté des variables regroupées. Un GBM trouve lui-même ses seuils, et lui donner les tranches du GLM pourrait le brider. La question sera tranchée en section 3 ;
- **la structure du fichier de tarif GLM**, pour écrire la fonction de scoring.

On affiche aussi les totaux par échantillon (exposition, sinistres, charge) : ce sont les quantités que tout tarif recalé doit reproduire.

In [1]:
# =============================================================================
# Section 1 : chargement, controles d'integrite, structure du tarif GLM
# =============================================================================
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Racine du projet : on remonte jusqu'au dossier qui contient .git,
# le notebook fonctionne ainsi quel que soit le dossier de lancement
ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
SEED = 2026

# --- 1. Donnees nettoyees et decoupage fige (relu, jamais regenere) ---
df = pd.read_parquet(DATA_DIR / "fremtpl2_clean.parquet")
split = pd.read_parquet(DATA_DIR / "split_train_test.parquet")

# validate="one_to_one" leve une erreur si une police apparait deux fois
df = df.merge(split[["IDpol", "split"]], on="IDpol", how="left", validate="one_to_one")
del split
gc.collect()

assert df["split"].notna().all(), "Des polices n'ont pas de split"
effectifs = df["split"].value_counts()
assert sorted(effectifs.tolist()) == [135_616, 542_397], f"Decoupage inattendu :\n{effectifs}"
print(f"Polices : {len(df):,}")
print(effectifs.to_string(), "\n")

# --- 2. Totaux par echantillon : ce que tout tarif recale doit reproduire ---
resume = df.groupby("split").agg(
    polices=("IDpol", "size"),
    exposition=("Exposure", "sum"),
    sinistres=("ClaimNb", "sum"),
    sinistres_sev=("ClaimNbSev", "sum"),
    charge=("ClaimAmount", "sum"),
)
resume["frequence"] = resume["sinistres"] / resume["exposition"]
resume["prime_pure_obs"] = resume["charge"] / resume["exposition"]
print(resume.round(4).to_string(), "\n")

# --- 3. Inventaire des colonnes ---
print(df.dtypes.to_string(), "\n")

# Variables brutes : candidates pour le GBM (decision en section 3)
COLS_BRUTES = ["DrivAge", "VehAge", "VehPower", "BonusMalus",
               "Density", "Region", "VehBrand", "VehGas"]
manquantes = [c for c in COLS_BRUTES if c not in df.columns]
print("Variables brutes manquantes :", manquantes if manquantes else "aucune", "\n")

# --- 4. Structure du tarif GLM v0.2.0 (cles et types, pas toutes les valeurs) ---
with open(MODELS_DIR / "tarif_glm_phase2.json", encoding="utf-8") as f:
    tarif_glm = json.load(f)


def apercu_structure(obj, niveau=0, max_elements=5):
    """Affiche l'arborescence d'un JSON : cles, types, tailles, quelques valeurs."""
    marge = "    " * niveau
    if isinstance(obj, dict):
        # Toutes les cles au premier niveau, un apercu ensuite
        limite = len(obj) if niveau == 0 else max_elements
        for i, (cle, val) in enumerate(obj.items()):
            if i == limite:
                print(f"{marge}... ({len(obj) - limite} cles de plus)")
                break
            if isinstance(val, (dict, list)):
                print(f"{marge}{cle} : {type(val).__name__} de {len(val)} elements")
                apercu_structure(val, niveau + 1, max_elements)
            else:
                print(f"{marge}{cle} : {val!r}")
    elif isinstance(obj, list):
        for val in obj[:max_elements]:
            if isinstance(val, (dict, list)):
                apercu_structure(val, niveau + 1, max_elements)
            else:
                print(f"{marge}{val!r}")
        if len(obj) > max_elements:
            print(f"{marge}... ({len(obj) - max_elements} elements de plus)")


apercu_structure(tarif_glm)

Polices : 678,013
split
train    542397
test     135616 

       polices   exposition  sinistres  sinistres_sev       charge  frequence  prime_pure_obs
split                                                                                        
test    135616   71579.5329       7096           5253  11525316.88     0.0991        161.0141
train   542397  286780.5725      28960          21191  48383899.62     0.1010        168.7140 

IDpol               int64
ClaimNb             int64
Exposure          float64
Area             category
VehPower            int64
VehAge              int64
DrivAge             int64
BonusMalus          int64
VehBrand         category
VehGas                str
Density             int64
Region           category
ClaimAmount       float64
ClaimNbSev          int64
DrivAgeBand      category
VehAgeBand       category
log_Density       float64
RegionGrouped         str
split                 str 

Variables brutes manquantes : aucune 

version : 'phase2-glm-v1'
dat

### Lecture des résultats

**Intégrité.** 678 013 polices, découpage identique à celui de la Phase 2 (542 397 / 135 616), aucune police sans échantillon. Les chiffres de référence du GLM restent comparables.

**Le test est un échantillon plus "léger", pas le signe d'une mauvaise calibration.** Pour un échantillon $s \in \{train, test\}$, on note $E_s$ l'exposition totale en années, $N_s$ le nombre de sinistres, $N_s^{sev}$ le nombre de sinistres dotés d'un montant et $C_s$ la charge totale. La prime pure observée se décompose exactement en fréquence × taux d'observation × coût moyen :

$$\frac{C_s}{E_s} = \frac{N_s}{E_s} \times \frac{N_s^{sev}}{N_s} \times \frac{C_s}{N_s^{sev}}$$

| Composante | Train | Test | Ratio test / train |
|---|---|---|---|
| Fréquence $N_s / E_s$ | 0.1010 | 0.0991 | 0.982 |
| Taux d'observation $N_s^{sev} / N_s$ | 0.7317 | 0.7403 | 1.012 |
| Coût moyen $C_s / N_s^{sev}$ (EUR) | 2 283 | 2 194 | 0.961 |
| Prime pure $C_s / E_s$ (EUR) | 168.71 | 161.01 | 0.954 |

L'écart de prime pure (-4.6 %) vient surtout du coût moyen (-3.9 %). C'est attendu : la Phase 2 a établi que la charge a une variance théoriquement infinie ($\xi \approx 0.8$), donc la moyenne de 5 253 montants n'est pas stabilisée et quelques gros sinistres suffisent à la déplacer de plusieurs pourcents. Côté fréquence, si $N_{test}$ suit une loi de Poisson, son écart-type relatif vaut $1/\sqrt{\mathbb{E}[N_{test}]} \approx 1/\sqrt{7096} \approx 1.2\,\%$ : l'écart de -1.8 % représente environ 1.5 écart-type, ce qui reste compatible avec le hasard.

**Lien avec l'A/E test de 0.9531.** On note $e_i$ l'exposition de la police $i$ et $\bar\pi_s = \frac{1}{E_s}\sum_{i \in s} e_i \widehat{PP}_i$ la prime pure moyenne prédite sur l'échantillon $s$, de sorte que $A/E_s = C_s / (E_s\,\bar\pi_s)$. Comme $A/E_{train} = 1$ par construction de $k$, on obtient :

$$\frac{\bar\pi_{test}}{\bar\pi_{train}} = \frac{C_{test}/E_{test}}{C_{train}/E_{train}} \times \frac{1}{A/E_{test}} = \frac{0.9544}{0.9531} \approx 1.0013$$

Le GLM juge le portefeuille de test aussi risqué que celui du train (+0.1 %). L'écart vient des réalisations, pas du modèle. Ce calcul suppose que l'A/E de la Phase 2 rapporte la charge brute totale à la charge prédite : point à vérifier en section 2.

**Conséquence pour le benchmark.** Tout modèle recalé sur le train affichera un A/E test proche de 0.95. L'A/E global ne départagera donc pas le GLM et le GBM : on comparera leur calibration par déciles de prime prédite.

**Variables.** Toutes les variables brutes sont présentes, donc le GBM pourra les recevoir sous forme continue (décision en section 3). Deux points techniques :
- `VehGas` et `RegionGrouped` sont stockées en `str`, le type texte par défaut de pandas 3. LightGBM le refuse : il faudra les convertir explicitement en `category`.
- `BonusMalusBand` et `VehPowerGrp` n'existent pas dans le parquet : elles ont été construites dans le notebook 02.

**Le tarif GLM.** Le fichier contient bien les 53 + 28 coefficients et les quatre constantes de la Phase 2. Contrôle de non-fuite : $\tau_{obs} = 0.7317334$ vaut exactement 21 191 / 28 960, le ratio du train, donc la constante a été calculée sans voir le test. En revanche, le fichier n'est pas autosuffisant : il référence `BonusMalusBand` et `VehPowerGrp` sans en donner les bornes. Or un tarif doit pouvoir se calculer à partir des seules variables brutes d'un assuré, et c'est exactement ce que fera l'appli en Phase 5. Enfin, `performance_test` fournit les cibles de la section 2 avec toutes leurs décimales : la reproduction devra être exacte, pas seulement approchée.

## 2. Reproduction du tarif GLM v0.2.0

Avant de mesurer le GBM, on vérifie que le code de mesure redonne les chiffres de la Phase 2. Si le GLM ne retrouve pas son propre Gini, aucun écart avec le GBM n'est interprétable : il pourrait venir du modèle comme de la mesure.

**Trois modules partagés**, importés ici et plus tard par l'appli :
- `src/features.py` construit toutes les variables tarifaires à partir des variables brutes : bornes des tranches, regroupements, nettoyage de `VehGas`. Une valeur hors bornes lève une erreur.
- `src/models/tarif_glm.py` recalcule le tarif à partir des coefficients du JSON, sans statsmodels. Une modalité absente du tarif lève une erreur, au lieu de recevoir silencieusement le tarif de la référence.
- `src/evaluation.py` regroupe les métriques. Elles ne prennent que des prédictions en entrée, jamais un modèle : GLM et GBM passent par exactement le même code.

**Notations.** Pour la police $i$ : $e_i$ son exposition en années, $N_i$ son nombre de sinistres, $S_i$ sa charge observée, $d_i$ la densité de sa commune et $m_{ij}$ sa modalité pour la variable catégorielle $j$.

**Prédicteur linéaire.** Pour une composante à lien log (fréquence ou sévérité), on note $\beta_0$ l'intercept, $\beta_{j,m}$ le coefficient de la modalité $m$ de la variable $j$ (nul pour la référence) et $\gamma$ le coefficient de $\log d_i$ :

$$\eta_i = \beta_0 + \sum_j \beta_{j,\,m_{ij}} + \gamma \log d_i \qquad \hat\lambda_i = e^{\eta_i^{freq}} \qquad \hat\mu_i^{ecr} = e^{\eta_i^{sev}}$$

L'offset $\log e_i$ n'apparaît pas : $\hat\lambda_i$ est la fréquence annuelle, et le nombre attendu de sinistres vaut $e_i \hat\lambda_i$.

**Contrôle structurel.** Le GLM Poisson à lien canonique avec intercept vérifie l'équation de score $\sum_{i \in train} (N_i - e_i \hat\lambda_i) = 0$. Un seul coefficient mal relu suffirait à casser cette égalité.

**Indice $p$ de la déviance de Tweedie.** Le JSON ne le contient pas. La Phase 2 l'a déduit du paramètre de forme $\alpha = 1/\phi$ de la loi Gamma, par $p = (\alpha + 2)/(\alpha + 1)$. On recalcule la dispersion $\phi$ par la statistique de Pearson, comme statsmodels. On note $n$ le nombre de polices de l'échantillon sévérité du train, $q = 28$ le nombre de paramètres, $w_i$ = `ClaimNbSev` le poids, $y_i$ le coût moyen écrêté et $\hat\mu_i$ sa prédiction :

$$\hat\phi = \frac{1}{n - q} \sum_{i=1}^{n} w_i \, \frac{(y_i - \hat\mu_i)^2}{\hat\mu_i^2}$$

**Modèles nuls des $D^2$.** $D^2 = 1 - D(\text{tarif}) / D(\text{nul})$. On conserve les conventions de la Phase 2 :

| Composante | Déviance | Modèle nul |
|---|---|---|
| Fréquence | Poisson sur $N_i / e_i$, poids $e_i$ | fréquence moyenne de l'échantillon évalué |
| Sévérité | Gamma sur le coût moyen écrêté, poids `ClaimNbSev` | coût moyen pondéré de l'échantillon évalué |
| Prime pure | Tweedie d'indice $p$ sur $S_i / e_i$, poids $e_i$ | prime pure moyenne du train |

**Deux changements de mesure par rapport à la Phase 2.**
- *Tri stable.* À primes égales, l'ordre des lignes est conservé. Le Gini ne dépend plus de l'algorithme de tri de la machine.
- *Groupes d'exposition égale.* La part cumulée d'exposition est bornée à 1. Le code de la Phase 2 ne le faisait pas, et un arrondi flottant pouvait exclure la police la plus chère du groupe G10. La partie 6 de la cellule mesure l'effet de ce changement sur le lift.

**Critère de reproduction.** Une valeur publiée à 4 décimales est reproduite si l'écart est inférieur à $5 \times 10^{-5}$.

In [5]:
# =============================================================================
# Section 2 : reproduction du tarif GLM v0.2.0 avec le code partage de src/
# =============================================================================
# autoreload : une modification d'un fichier de src/ est prise en compte
# sans redemarrer le kernel
%reload_ext autoreload
%autoreload 2

import sys

# Rend le dossier src/ importable depuis le notebook
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import evaluation as ev
from src.features import echantillon_severite, prepare_variables
from src.models.tarif_glm import (charge_tarif, cout_moyen_ecrete,
                                  frequence_annuelle, prime_pure)

# --- 1. Variables tarifaires recalculees depuis les variables brutes ---
# Controle : src/features.py doit redonner exactement les colonnes de la Phase 1
anciennes = df[["DrivAgeBand", "VehAgeBand", "RegionGrouped", "log_Density"]].copy()
df = prepare_variables(df)
for col in anciennes.columns:
    identique = np.array_equal(df[col].astype(str).to_numpy(),
                               anciennes[col].astype(str).to_numpy())
    print(f"{col:<14} identique a la Phase 1 : {identique}")
    assert identique, f"{col} : src/features.py ne reproduit pas la Phase 1"
del anciennes
print(f"VehGas apres nettoyage : {sorted(df['VehGas'].unique())}\n")

# --- 2. Scoring du tarif depuis le JSON, sur tout le portefeuille ---
tarif = charge_tarif(MODELS_DIR / "tarif_glm_phase2.json")
C = tarif["constantes"]

df["freq_glm"] = frequence_annuelle(tarif, df)   # lambda_i
df["sev_glm"] = cout_moyen_ecrete(tarif, df)     # mu_i (cout ecrete)
df["pp_glm"] = prime_pure(tarif, df)             # PP_i

train = df[df["split"] == "train"]
test = df[df["split"] == "test"]

# --- 3. Controle structurel : equilibre du GLM Poisson sur le train ---
# Avec un lien canonique et un intercept, la somme des sinistres predits egale
# la somme des sinistres observes. Un coefficient mal relu casserait l'egalite.
ratio_eq = (train["freq_glm"] * train["Exposure"]).sum() / train["ClaimNb"].sum()
print(f"Sinistres predits / observes (train) : {ratio_eq:.10f}")
assert abs(ratio_eq - 1) < 1e-8

# --- 4. Echantillons severite et indice p de la deviance de Tweedie ---
sev_train = echantillon_severite(train, C["seuil_ecretement_eur"])
sev_test = echantillon_severite(test, C["seuil_ecretement_eur"])

# Dispersion du GLM Gamma (statistique de Pearson) : n observations, q parametres
w = sev_train["ClaimNbSev"].to_numpy()
y = sev_train["cout_ecrete"].to_numpy()
mu = sev_train["sev_glm"].to_numpy()
n_obs, q = len(sev_train), len(tarif["brut"]["severite"]["coefficients"])
PHI_SEV = np.sum(w * (y - mu) ** 2 / mu ** 2) / (n_obs - q)
ALPHA = 1 / PHI_SEV
P_TWEEDIE = (ALPHA + 2) / (ALPHA + 1)

# Modele nul du D2 Tweedie : prime pure uniforme calibree sur le train
PP_NULLE = train["ClaimAmount"].sum() / train["Exposure"].sum()

print(f"\nEchantillons severite : {n_obs:,} (train) | {len(sev_test):,} (test)")
print(f"Dispersion phi = {PHI_SEV:.4f} | alpha = {ALPHA:.4f} | p = {P_TWEEDIE:.6f}")
print(f"Prime pure du modele nul : {PP_NULLE:.4f} EUR\n")

# --- 5. Les indicateurs du tarif, recalcules par src/evaluation.py ---
_, _, gini = ev.lorenz(test["pp_glm"], test["Exposure"], test["ClaimAmount"])
lift = ev.tableau_lift(test["pp_glm"], test["Exposure"],
                       test["ClaimNb"], test["ClaimAmount"])

perf_p2 = tarif["brut"]["performance_test"]
reproduction = pd.DataFrame({
    "phase_2": [
        perf_p2["gini"],
        perf_p2["lift_frequence"],
        0.0357, 0.0034, 0.0519,
        1.0000,
        perf_p2["ae_test"],
        11.9003,
    ],
    "phase_3": [
        gini,
        lift.loc["G10", "frequence"] / lift.loc["G1", "frequence"],
        ev.d2_poisson(test["ClaimNb"], test["Exposure"], test["freq_glm"]),
        ev.d2_gamma(sev_test["cout_ecrete"], sev_test["ClaimNbSev"], sev_test["sev_glm"]),
        ev.d2_tweedie(test["ClaimAmount"], test["Exposure"], test["pp_glm"],
                      P_TWEEDIE, PP_NULLE),
        ev.ratio_ae(train["ClaimAmount"], train["Exposure"], train["pp_glm"]),
        ev.ratio_ae(test["ClaimAmount"], test["Exposure"], test["pp_glm"]),
        ev.amplitude(test["pp_glm"]),
    ],
}, index=["Gini", "Lift G10/G1 frequence", "D2 frequence", "D2 severite",
          "D2 prime pure", "A/E train", "A/E test", "Amplitude 99/1"])

# Valeurs de la Phase 2 publiees a 4 decimales : ecart tolere d'une
# demi-unite du dernier chiffre (5e-5)
reproduction["ecart"] = reproduction["phase_3"] - reproduction["phase_2"]
reproduction["reproduit"] = reproduction["ecart"].abs() < 5e-5
print(reproduction.to_string(float_format=lambda x: f"{x:.6f}"))

# --- 6. Effet de la correction du regroupement par exposition egale ---
# Le code de la Phase 2 calculait la part cumulee d'exposition sans la borner
# a 1. On reproduit son calcul pour voir ce qu'il faisait de la derniere police.
ordre = np.argsort(test["pp_glm"].to_numpy(), kind="stable")
part_cum = np.cumsum(test["Exposure"].to_numpy()[ordre]) / test["Exposure"].sum()
groupe_p2 = pd.cut(part_cum[-1:], bins=np.linspace(0, 1, 11), include_lowest=True)
print(f"\nPart cumulee de la derniere police : {float(part_cum[-1])!r}")
print(f"Groupe attribue par le code de la Phase 2 : {groupe_p2[0]}")

plus_chere = test.index[ordre[-1]]
nb_pc, expo_pc = test.loc[plus_chere, "ClaimNb"], test.loc[plus_chere, "Exposure"]
print(f"Police concernee : prime {test.loc[plus_chere, 'pp_glm']:,.0f} EUR, "
      f"exposition {expo_pc:.2f}, {nb_pc} sinistre(s)")

# Lift recalcule en retirant cette police du groupe G10
freq_g10_p2 = (lift.loc["G10", "nb"] - nb_pc) / (lift.loc["G10", "expo"] - expo_pc)
print(f"Lift sans cette police : {freq_g10_p2 / lift.loc['G1', 'frequence']:.6f}")

DrivAgeBand    identique a la Phase 1 : True
VehAgeBand     identique a la Phase 1 : True
RegionGrouped  identique a la Phase 1 : True
log_Density    identique a la Phase 1 : True
VehGas apres nettoyage : ['Diesel', 'Regular']

Sinistres predits / observes (train) : 1.0000000000

Echantillons severite : 20,010 (train) | 4,934 (test)
Dispersion phi = 2.0058 | alpha = 0.4986 | p = 1.667306
Prime pure du modele nul : 168.7140 EUR

                        phase_2   phase_3     ecart  reproduit
Gini                   0.359234  0.359233 -0.000000       True
Lift G10/G1 frequence  4.522628  4.524836  0.002209      False
D2 frequence           0.035700  0.035666 -0.000034       True
D2 severite            0.003400  0.003357 -0.000043       True
D2 prime pure          0.051900  0.051890 -0.000010       True
A/E train              1.000000  1.000000  0.000000       True
A/E test               0.953130  0.953130  0.000000       True
Amplitude 99/1        11.900300 11.900296 -0.000004       True



### Lecture des résultats

**Le code partagé reproduit le tarif v0.2.0.** Sept indicateurs sur huit sont retrouvés dans la tolérance, et l'écart sur le lift est entièrement expliqué ci-dessous. Chaque contrôle valide une brique différente :

| Contrôle | Résultat | Ce qu'il valide |
|---|---|---|
| Variables recalculées par `features.py` | identiques sur les 678 013 polices | bornes et regroupements de la Phase 1 |
| Équilibre $\sum_i e_i \hat\lambda_i \,/\, \sum_i N_i$ (train) | 1.0000000000 | les 53 coefficients de fréquence et leur décodage |
| Dispersion $\hat\phi$ du GLM Gamma | 2.0058 (Phase 2 : 2.006) | les 28 coefficients de sévérité et l'échantillon sévérité |
| A/E test | 0.9531295, identique au JSON | les quatre constantes et la recombinaison |
| $D^2$ fréquence, sévérité, prime pure | écarts inférieurs à $5 \times 10^{-5}$ | les fonctions de déviance et les conventions de modèle nul |

L'équilibre est le contrôle le plus exigeant. Il ne tient que si la fréquence prédite est exactement celle du GLM ajusté : un coefficient attribué à la mauvaise modalité l'aurait cassé.

L'A/E test identique confirme aussi l'hypothèse de la section 1 : l'A/E de la Phase 2 rapporte bien la charge brute totale à la charge prédite.

**Gini : un écart d'environ $4 \times 10^{-7}$, dû aux ex aequo.** Deux polices de même commune et de mêmes tranches ont exactement la même prime. Leur ordre relatif dans le tri est arbitraire, et il déplace légèrement la courbe de Lorenz puisque leurs charges diffèrent. La Phase 2 utilisait un tri instable, dont l'ordre entre ex aequo dépend de l'implémentation. Le tri stable conserve l'ordre des lignes. L'écart porte sur la 7e décimale : il est sans portée pratique, mais la mesure est désormais déterministe.

**Lift : un défaut du code de la Phase 2, corrigé.** La part cumulée d'exposition de la dernière police vaut 1.0000000000001275, et non 1. Cela vient de l'arithmétique flottante : `np.cumsum` additionne les 135 616 expositions une à une, tandis que `.sum()` les additionne par paires, et les deux calculs n'accumulent pas les mêmes erreurs d'arrondi. Hors de l'intervalle $]0.9,\, 1]$, `pd.cut` renvoie `NaN`, et le `groupby(observed=True)` écarte la police sans message.

La police écartée est la plus chère du test : 2 650 EUR de prime, un sinistre en un an d'exposition. Si on la retire du groupe G10, on retrouve 4.522628, soit exactement la valeur de la Phase 2. L'écart est donc entièrement expliqué, et le lift corrigé du GLM vaut **4.5248**.

Trois conséquences :
- *Les conclusions de la Phase 2 tiennent.* Une police sur environ 7 158 années-police du groupe G10 ne change ni le lift arrondi (4.52) ni la monotonie.
- *Le défaut aurait faussé le benchmark.* Il frappe toujours la police la plus chère, donc celle du groupe où les modèles diffèrent le plus. Pour un GBM aux primes plus extrêmes, la police écartée aurait pu porter un sinistre grave et modifier la charge observée de G10.
- *Le tableau de calibration de la section 4.3 du notebook 02* utilisait le même calcul et avait le même défaut, avec le même effet négligeable.

**Référence du benchmark.** Les valeurs de la colonne `phase_3` deviennent la référence GLM pour toute la suite :

| Indicateur (test) | GLM v0.2.0 |
|---|---|
| Gini | 0.3592 |
| Lift G10/G1 fréquence | 4.5248 |
| $D^2$ fréquence | 0.0357 |
| $D^2$ sévérité | 0.0034 |
| $D^2$ prime pure ($p = 1.667306$) | 0.0519 |
| A/E train / test | 1.0000 / 0.9531 |
| Amplitude 99/1 | 11.90 |

L'indice $p = 1.667306$ est figé pour toute la phase. La déviance de Tweedie n'est comparable qu'à indice égal : le GBM sera jugé avec la même valeur, même s'il est ajusté avec un objectif Tweedie d'un autre indice.